In [ ]:
import scanpy as sc 
import pandas as pd 
import numpy as np 

import matplotlib.pyplot as plt 
import seaborn as sns 

from sklearn.metrics import adjusted_rand_score
from sklearn.decomposition import PCA

import scipy.sparse as sp 
import warnings

warnings.filterwarnings("ignore")

import os
import ctypes
import sys

# 1. 先设置 R_HOME
os.environ["R_HOME"] = "/home/pxy/miniconda3/envs/r40/lib/R"

# 2. 【核心黑科技】手动加载 R 的动态库
# 这步操作等同于在终端里设置 LD_LIBRARY_PATH，专门解决 VS Code 找不到库的问题
try:
    # 这是 R 的核心库路径
    libR_path = "/home/pxy/miniconda3/envs/r40/lib/R/lib/libR.so"
    # 强制加载进内存
    ctypes.CDLL(libR_path, mode=ctypes.RTLD_GLOBAL)
    print("✅ 成功强制加载 libR.so")
except OSError as e:
    print(f"❌ 加载失败: {e}")

# 3. 然后再导入其他包
sys.path.append("..") 

import spCLUE
import rpy2.robjects as robjects
print("R 环境路径:", robjects.r['R.home']()[0])

spCLUE.fix_seed(0)

# 定义DLPFC数据集的12个切片ID
slice_ids = [
    "151507", "151508", "151509", "151510",
    "151669", "151670", "151671", "151672",
    "151673", "151674", "151675", "151676"
]

# 用于存储每个切片的ARI结果
ari_results = []

# 数据路径（请根据实际情况确认路径是否正确）
data_dir = '/home/pxy/home/pxy/data/DLPFC/st/'

print(f"Start processing {len(slice_ids)} slices...")

for sample_name in slice_ids:
    print(f"\n{'='*20} Processing Sample: {sample_name} {'='*20}")
    
    # 1. 设置簇的数量 (根据DLPFC数据集的已知Ground Truth)
    # 151669-151672 通常只有5层，其他切片为7层
    if sample_name in ["151669", "151670", "151671", "151672"]:
        n_clusters = 5
    else:
        n_clusters = 7
    
    try:
        # 2. 加载数据
        # 使用 read_visium 加载数据，路径拼接逻辑参考原文件
        adata = sc.read_visium(data_dir + sample_name)
        adata.var_names_make_unique()
        
        # 加载元数据 (Ground Truth)
        meta = pd.read_csv(data_dir + sample_name + "/metadata.tsv", sep="\t")
        meta = meta.set_index("barcode")
        adata.obs["Region"] = meta.loc[adata.obs_names, "layer_guess_reordered"]
        
        # 3. 数据预处理与构图
        # 原文件 Cell 6 的逻辑
        adata = spCLUE.preprocess(adata, n_top_genes=3000)
        adata.obsm["X_pca"] = PCA(n_components=200, random_state=0).fit_transform(adata.X)
        
        g_spatial = spCLUE.prepare_graph(adata, "spatial", n_neighbors=8)
        g_expr = spCLUE.prepare_graph(adata, "expr", n_neighbors=8)
        graph_dict = {"spatial": g_spatial, "expr": g_expr}
        
        # 4. 模型初始化与训练
        # 原文件 Cell 8 的逻辑
        # 注意：这里将 n_clusters 参数改为动态变量，与当前切片保持一致
        spCLUE_model = spCLUE.spCLUE(
            input_data=adata.obsm['X_pca'],
            hvg_input=adata.obsm['X_pca'],
            graph_dict=graph_dict,
            n_clusters=n_clusters,
            reconstruction_loss='mse',
            gamma=1.0,
            gamma_mask=0.5,
            beta=1.0,
            kappa=5.0,
            # cluster_weight_strategy='min',  # 🔥 置信度策略
            # cluster_warmup_epochs=50,       # 🔥 Warmup
            # dim_input=2000,
            # dim_hidden=256,
            dim_embed=16,
            graph_corr=0.4,
            epochs = 800,
            patience=50,
            min_delta=0.005,
        )
  
        # 训练模型
        _, adata.obsm["spCLUE"] = spCLUE_model.train()
        
        # 5. 聚类
        # 原文件 Cell 10 的逻辑
        refinement = True
        cluster_method = "mclust"
        spCLUE.clustering(
            adata,
            n_clusters,
            key="spCLUE",
            refinement=refinement,
            cluster_methods=cluster_method,
        )
        
        # 6. 计算 ARI
        # 原文件 Cell 12 的逻辑
        # 过滤掉 Ground Truth 为 NA 的区域
        adata_valid = adata[adata.obs.Region.notna()]
        ARI = adjusted_rand_score(adata_valid.obs["Region"], adata_valid.obs["mclust_refined"])
        
        print(f"Sample {sample_name} ARI: {ARI:.8f}")
        ari_results.append(ARI)
        
    except Exception as e:
        print(f"Error processing sample {sample_name}: {e}")

# 7. 输出最终统计结果
print(f"\n{'='*20} Final Results {'='*20}")
if ari_results:
    mean_ari = np.mean(ari_results)
    median_ari = np.median(ari_results)
    print(f"ARI per slice: {[round(x, 4) for x in ari_results]}")
    print(f"Mean ARI: {mean_ari:.4f}")
    print(f"Median ARI: {median_ari:.4f}")
else:
    print("No ARI results collected.")

✅ 成功强制加载 libR.so
R 环境路径: /home/pxy/miniconda3/envs/r40/lib/R
Start processing 12 slices...

==================== Processing Sample: 151507 ====================

预处理单切片数据 (全基因 PCA + 归一化 HVG)

根据原始计数挑选 top 3000 个高变基因...
全量数据归一化 (target_sum=1e4) & Log1p...
  ✓ 已保存归一化后的 HVG 重构目标: (4226, 3000)
将全量稀疏矩阵转换为稠密矩阵以计算 PCA...
计算 PCA (基于全基因归一化数据, n_components=200)...

✅ 预处理完成！
  - adata.X (全量归一化): (4226, 11982)
  - adata.obsm['X_hvg'] (归一化HVG): (4226, 3000)
  - adata.obsm['X_pca'] (全基因PCA): (4226, 200)

构建 spatial 图...
  - 使用空间坐标: (4226, 2)
  ✓ 空间图构建完成:  (4226, 4226), 边数=35800

构建 expr 图...
  - 使用PCA:  (4226, 200)
  ✓ 表达图构建完成: (4226, 4226), 边数=65578
ℹ️ 初始化单切片模型 (CCGCN)
   - 输入:  200维PCA
   - 重构: 2000个HVG
✅ 使用 MSE 重构损失
Training Start =========================>
重构损失类型: MSE
聚类对齐策略: min
Warmup epochs: 50
早停策略: patience=50, min_delta=0.005
损失权重: kappa=5.0, beta=1.0, gamma=1.0


  1%|          | 5/800 [00:01<02:37,  5.04it/s]


Epoch 1:   Loss=13.3788, ARI=0.054
 Cluster=2.5709, Recon=2.5869, MaskRecon=2.5877, LocalAgg=0.6927
  ClusterLoss=2.5627, NegLoss=0.0082


  7%|▋         | 57/800 [00:02<00:21, 34.53it/s]


Epoch 50:   Loss=12.1210, ARI=0.459
 Cluster=2.3451, Recon=2.5685, MaskRecon=2.5697, LocalAgg=0.5923
  ClusterLoss=2.3446, NegLoss=0.0005


...


Epoch 300:   Loss=9.9385, ARI=0.540
 Cluster=0.6309, Recon=2.4827, MaskRecon=2.4601, LocalAgg=0.5595
  ClusterLoss=0.6257, NegLoss=0.0052


 44%|████▍     | 355/800 [00:09<00:12, 35.66it/s]


Epoch 350:   Loss=9.7476, ARI=0.545
 Cluster=0.5603, Recon=2.4812, MaskRecon=2.4535, LocalAgg=0.5479
  ClusterLoss=0.5538, NegLoss=0.0065


 45%|████▍     | 358/800 [00:09<00:12, 36.40it/s]



Early stopping at epoch 359
Best loss: 9.6134
Training Finished =================<
fitting ...
  |======================================================================| 100%
Sample 151676 ARI: 0.4303

==================== Final Results ====================
ARI per slice: [0.5779, 0.4587, 0.4717, 0.4652, 0.3417, 0.2527, 0.8199, 0.5944, 0.5408, 0.4307, 0.5204, 0.4303]
Mean ARI: 0.4920
Median ARI: 0.4685
